# Eksploracja: ekstrakcja tekstu z PDFf
Ten nootbook ma na celu sprawdzenie w jaki sposób wydobyć treść pdf aby była ona przystępna dla LLM-a.

## 1. Setup

In [ ]:
import re
from pathlib import Path

import pymupdf

pdf_path = Path.cwd().parent / "data" / "raport_2024_wybrane_strony.pdf"
print(f"PDF: {pdf_path}")
print(f"Istnieje: {pdf_path.exists()}")

## 2. Otwieranie dokumentu — co dostajemy?

In [ ]:
doc = pymupdf.open(str(pdf_path))

print(f"Typ obiektu: {type(doc)}")
print(f"Liczba stron: {len(doc)}")
print(f"\nMetadane dokumentu:")
for key, val in doc.metadata.items():
    if val:
        print(f"  {key}: {val}")

## 3. Pojedyncza strona — inspekcja obiektu Page

In [ ]:
page = doc[3]  # indeks 0-based → to strona numer 2 w PDF

print(f"Typ: {type(page)}")
print(f"Numer strony (0-based): {page.number}")
print(f"\nMetody i właściwości (bez __dunder__):")
public_attrs = [a for a in dir(page) if not a.startswith("_")]
print(", ".join(public_attrs))


## 4. Tryby ekstrakcji tekstu — porównanie

`get_text()` przyjmuje argument który zmienia format wyjścia.\
Sprawdzamy 4 tryby na tej samej stronie — każdy daje inne możliwości.

In [ ]:
# Tryb "text" — domyślny, ciągły tekst ze znakami nowej linii
# To właśnie używamy w naive_processor.py
text_plain = page.get_text("text")
print("=== TRYB 'text' (domyślny) ===")
print(f"Typ: {type(text_plain)}, długość: {len(text_plain)} znaków")
print(text_plain[:400])

In [ ]:
# Tryb "blocks" — tekst podzielony na bloki z pozycjami na stronie
# Każdy blok: (x0, y0, x1, y1, text, block_no, block_type)
# block_type: 0 = tekst, 1 = obraz
blocks = page.get_text("blocks")
print("=== TRYB 'blocks' ===")
print(f"Liczba bloków: {len(blocks)}")
print(f"Struktura pierwszego bloku: {blocks[0]}")
print(f"\nPierwsze 5 bloki tekstowe:")
for b in blocks[:]:
    x0, y0, x1, y1, text, block_no, block_type = b
    kind = "tekst" if block_type == 0 else "obraz"
    print(f"  [{kind}] pos=({x0:.0f},{y0:.0f})-({x1:.0f},{y1:.0f})  '{text[:].strip()}'")


In [ ]:
# Tryb "words" — każde słowo osobno z pozycją
# Każde słowo: (x0, y0, x1, y1, word, block_no, line_no, word_no)
words = page.get_text("words")
print("=== TRYB 'words' ===")
print(f"Liczba słów: {len(words)}")
print(f"Struktura pierwszego słowa: {words[0]}")
print(f"\nPierwsze 10 słów:")
for w in words[:10]:
    print(f"  '{w[4]}' @ ({w[0]:.0f},{w[1]:.0f})")

In [ ]:
# Tryb "dict" — pełna hierarchia: bloki → linie → spany
# Span zawiera: tekst, czcionkę, rozmiar, kolor, flagi (bold/italic)
d = page.get_text("dict")
print("=== TRYB 'dict' ===")
print(f"Klucze: {list(d.keys())}")
print(f"Liczba bloków: {len(d['blocks'])}")

# Wejdź głębiej w pierwszy blok tekstowy
first_text_block = next(b for b in d["blocks"] if b.get("type") == 0)
print(f"\nPierwszy blok tekstowy:")
print(f"  Liczba linii: {len(first_text_block['lines'])}")
first_line = first_text_block["lines"][0]
print(f"  Pierwsza linia, liczba spanów: {len(first_line['spans'])}")
first_span = first_line["spans"][0]
print(f"  Pierwszy span:")
print(f"    tekst:    '{first_span['text']}'")
print(f"    font:     {first_span['font']}")
print(f"    rozmiar:  {first_span['size']:.1f} pt")
print(f"    flagi:    {first_span['flags']} (bit 4=bold, bit 1=italic)")

**Podsumowanie trybów:**

| Tryb | Zwraca | Kiedy używać |
|---|---|---|
| `"text"` | `str` — ciągły tekst | Prosta ekstrakcja, RAG chunking |
| `"blocks"` | `list[tuple]` — bloki z pozycjami | Gdy potrzebujesz wiedzieć *gdzie* jest tekst na stronie |
| `"words"` | `list[tuple]` — słowa z pozycjami | Rekonstrukcja tabel po współrzędnych |
| `"dict"` | `dict` — pełna hierarchia + metadane fontów | Wykrywanie nagłówków (duży font), bold, struktury |

**Jedynym trybem pozwalającym na wierne odwzorowanie dokumentu jest `"dict"`.**
Tylko on daje jednocześnie: pozycję każdego fragmentu tekstu na stronie (`bbox`), krój i rozmiar czcionki, kolor tekstu oraz informację o bold/italic.
Bez tych metadanych przy RAG wszystko się rozjeżdża — nawigacja BGK miesza się z treścią rozdziałów, kolumny tabelaryczne tracą układ, a nagłówki sekcji są nie do odróżnienia od zwykłego tekstu.
Tryb `"text"` jest wystarczający tylko gdy PDF jest jednokolumnowy, nie zawiera tabel i nie ma elementów nawigacyjnych — czyli praktycznie nigdy w przypadku raportów finansowych.

## 4b. pdfplumber — sprawdzenie alternatyw

`pdfplumber` to druga popularna biblioteka do PDF w Pythonie, zbudowana na bazie `pdfminer.six`.
Kluczowa różnica wobec PyMuPDF: zamiast trybu ekstrakcji wybieranego flagą, każdy rodzaj danych to osobna metoda.
Poniżej ten sam schemat eksploracji co w sekcji 4 — obiekt strony, metody ekstrakcji, analiza jakości.

In [ ]:
import pdfplumber

pdf_pl = pdfplumber.open(str(pdf_path))

print(f"Typ obiektu: {type(pdf_pl)}")
print(f"Liczba stron: {len(pdf_pl.pages)}")
print(f"\nMetadane:")
for key, val in pdf_pl.metadata.items():
    if val:
        print(f"  {key}: {val}")

### Inspekcja obiektu Page

In [ ]:
page_pl = pdf_pl.pages[3]

print(f"Typ: {type(page_pl)}")
print(f"Numer strony (0-based): {page_pl.page_number}")
print(f"Wymiary: szerokość={page_pl.width:.1f} pt, wysokość={page_pl.height:.1f} pt")
print(f"\nMetody i właściwości (bez __dunder__):")
public_attrs = [a for a in dir(page_pl) if not a.startswith("_")]
print(", ".join(public_attrs))

### Metody unikalne dla pdfplumber

`extract_text()`, `extract_words()` i `chars` mają odpowiedniki w PyMuPDF — pomijamy.
Poniżej tylko to czego PyMuPDF nie potrafi:

| Metoda | Zwraca | Co daje |
|---|---|---|
| `extract_tables()` | `list[list[list[str]]]` — tabele → wiersze → komórki | Automatyczna rekonstrukcja układu tabelarycznego |
| `find_tables()` | `list[Table]` — obiekty z `bbox` | Pozycja tabeli na stronie; pozwala wyciąć jej obszar z `extract_text()` |

In [ ]:
# extract_tables() — automatyczna rekonstrukcja tabel
# Zwraca list[list[list[str]]] — tabele → wiersze → komórki
# To jest to czego PyMuPDF "text" nie potrafi
print("=== extract_tables() — skan wszystkich stron ===")
for i, p in enumerate(pdf_pl.pages, start=1):
    tables = p.extract_tables()
    if tables:
        print(f"\nStrona {i}: {len(tables)} tabela(e)")
        for t_idx, table in enumerate(tables):
            print(f"  Tabela {t_idx+1}: {len(table)} wierszy × {len(table[0])} kolumn")
            for row in table[:3]:  # pierwsze 3 wiersze
                print(f"    {row}")
            if len(table) > 3:
                print(f"    ... ({len(table)-3} wierszy więcej)")

In [ ]:
# find_tables() — obiekty tabel z bbox (gdzie fizycznie leżą na stronie)
# Pozwala wykluczyć obszar tabeli z extract_text() żeby uniknąć duplikacji
print("=== find_tables() — pozycje tabel na stronie ===")
for i, p in enumerate(pdf_pl.pages, start=1):
    found = p.find_tables()
    if found:
        print(f"\nStrona {i}: {len(found)} tabela(e)")
        for t in found:
            print(f"  bbox: {tuple(round(x,1) for x in t.bbox)}")

In [ ]:
pdf_pl.close()

### Wnioski z 4b

**pdfplumber nie nadaje się do tego dokumentu.**

`extract_tables()` teoretycznie rekonstruuje układ komórek, ale na praktycznie każdej stronie raportu BGK kolejność tekstu jest zepsuta — nawigacja boczna, nagłówki kolumn i treść akapitów mieszają się bez żadnej sensownej kolejności.
`extract_text()` i `extract_tables()` działają niezależnie i nie ma mechanizmu który złożyłby je z powrotem we właściwej kolejności.

Wynik: dla RAG bezużyteczny — chunki zawierają liczby bez narracyjnego kontekstu, albo fragmenty zdań w losowej kolejności.
PyMuPDF z trybem `"dict"` + filtrowanie po `bbox` daje więcej kontroli przy mniejszym chaosie.

## 4c. MarkItDown — konwersja PDF → Markdown

`markitdown` to biblioteka Microsoftu która konwertuje PDF (i inne formaty: DOCX, PPTX, XLSX) do Markdown.
Inne podejście niż PyMuPDF i pdfplumber — zamiast dawać surowe dane, interpretuje strukturę i od razu produkuje tekst gotowy do indeksowania.

Potencjalna zaleta dla RAG: tabele wychodzą jako tabele Markdown (`| col | col |`), nagłówki jako `#` — LLM rozumie tę składnię lepiej niż ciąg liczb bez kontekstu.

Ograniczenie: jeden wynik dla całego dokumentu, bez granularnego dostępu per strona.

In [ ]:
from markitdown import MarkItDown

md = MarkItDown()
result = md.convert(str(pdf_path))

print(f"Typ wyniku: {type(result)}")
print(f"Atrybuty: {[a for a in dir(result) if not a.startswith('_')]}")
print(f"\nDługość tekstu: {len(result.text_content)} znaków")

### Podgląd surowego wyjścia

In [ ]:
# Pierwsze 1000 znaków — czy wychodzi poprawny Markdown?
print(result.text_content[:1000])

In [ ]:
from IPython.display import Markdown, display

display(Markdown(result.text_content))

In [ ]:
# Czy są tabele Markdown w wyjściu?
lines = result.text_content.split("\n")
table_lines = [l for l in lines if l.strip().startswith("|")]

print(f"Łączna liczba linii: {len(lines)}")
print(f"Linie wyglądające jak tabela Markdown (zaczynają się od '|'): {len(table_lines)}")

if table_lines:
    print("\nPrzykład — pierwsze 20 linii tabelarycznych:")
    for l in table_lines[:20]:
        print(l)
else:
    print("\nBrak tabel Markdown w wyjściu.")

In [ ]:
# Problem: MarkItDown zwraca jeden blok tekstu — brak numerów stron
# Symulujemy podział na "strony" przez wyszukanie markerów podziału strony
# MarkItDown czasem wstawia <!-- Page N --> lub \f (form feed)
import re

page_markers = [i for i, l in enumerate(lines) if re.search(r'page|strona|\f', l, re.IGNORECASE)]
ff_count = result.text_content.count("\f")  # form feed = znak podziału strony

print(f"Znaki podziału strony (\\f): {ff_count}")
print(f"Linie z 'page'/'strona': {len(page_markers)}")

if ff_count > 0:
    pages_md = result.text_content.split("\f")
    print(f"\nDokument podzielony na {len(pages_md)} stron przez \\f")
    print(f"Pierwsza strona ({len(pages_md[0])} znaków):")
    print(pages_md[0][:400])
else:
    print("\nBrak markerów stron — MarkItDown zwraca jeden ciągły blok tekstu.")

In [ ]:
# Wydruk każdej strony osobno (podział po \f)
if ff_count > 0:
    for i, page_text in enumerate(pages_md, start=1):
        sep = "=" * 60
        print(f"\n{sep}")
        print(f"STRONA {i}  |  {len(page_text)} znaków")
        print(sep)
        print(page_text)
else:
    print("Brak \\f — wydruk pełnego tekstu jako jeden blok:")
    print(result.text_content)

### Wnioski z 4c

**MarkItDown nie radzi sobie z tym dokumentem.**

- Tabele finansowe nie zostały wykryte — brak jakichkolwiek linii `|...|` w wyjściu
- Nagłówki sekcji nie są oznaczone jako `#` — struktura dokumentu jest płaska

## Podsumowanie sekcji 4 — wybór narzędzia

Przetestowałem trzy biblioteki na tym samym dokumencie:

| Biblioteka | Tabele | Numery stron | Pozycje bbox | Kolory / fonty | Kolejność tekstu | Wynik |
|---|---|---|---|---|---|---|
| PyMuPDF `"text"` | nie | tak | nie | nie | rozjeżdża się | za mało danych |
| PyMuPDF `"dict"` | ręcznie | tak | tak | tak | kontrolowana | **wybrano** |
| pdfplumber | automatycznie | tak | tak | nie | zepsuta na każdej stronie | odpada |
| MarkItDown | nie wykrywa | nie | nie | nie | płaski tekst | odpada |

**Dalsza praca oparta będzie o PyMuPDF w trybie `"dict"`.**

Powody:

1. **Numery stron są pewne** — `page.number` per strona, bez heurystyk. Cytowania `[Strona N]` w RAG wymagają tego bezwzględnie.
2. **Pełna kontrola nad kolejnością** — `bbox` każdego spana pozwala samodzielnie zdecydować co jest treścią główną, a co nawigacją boczną lub stopką. pdfplumber i MarkItDown tej decyzji nie dają.
3. **Metadane fontu i koloru** — rozmiar czcionki pozwala wykryć nagłówki, kolor pozwala odróżnić strony działowe od treści. Żadna inna testowana biblioteka tego nie dostarcza.
4. **Tabele finansowe** — tryb `"words"` + rekonstrukcja po współrzędnej Y to więcej pracy niż `extract_tables()`, ale daje przewidywalny wynik bez psucia kolejności całej reszty strony.

Tryb `"text"` był punktem wyjścia w `naive_processor.py` — działa, ale traci wszystkie metadane. Kolejne eksperymenty będą sprawdzać czy filtrowanie po `bbox` i rozmiarze fontu i kolorze z trybu `"dict"` realnie poprawia jakość chunków trafiających do ChromaDB.